# Phase 3b: Macro Data Preprocessing — Industry Vacancies

This notebook retrieves and cleans the BFS dataset on job vacancies by economic division (NOGA classification) via the PXWeb API. The output is used in the Phase 3 micro-macro merge as the industry vacancy benchmark.

**Input:** BFS PXWeb API — `px-x-0602000000_103`  
**Output:** `data/processed/bfs_vacancies_economic_division_clean.csv`

---

## Table of Contents
1. [Environment Setup](#step1)
2. [API Request](#step2)
3. [Convert JSON-stat to DataFrame](#step3)
4. [Save Raw Dataset](#step4)
5. [Data Cleaning](#step5)
6. [Filter to Recent Quarters](#step6)
7. [Validation](#step7)
8. [Export Clean Dataset](#step8)

### Step 1: Environment Setup <a id="step1"></a>

Required libraries are imported and project directory paths are defined. Output folders are created automatically if they do not exist.

In [4]:
import requests
import pandas as pd
import itertools
from pathlib import Path

# Project paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

### Step 2: API Request <a id="step2"></a>
The BFS PXWeb API is queried for all available vacancy observations by economic division. The response is returned in JSON-stat2 format.

In [18]:
# API Request — explicitly request industry breakdown and total vacancies

API_URL = "https://www.pxweb.bfs.admin.ch/api/v1/en/px-x-0602000000_103/px-x-0602000000_103.px"

query = {
    "query": [
        {
            "code": "Offene Stellen",
            "selection": {
                "filter": "item",
                "values": ["1"]  # Job vacancies - total only
            }
        },
        {
            "code": "Wirtschaftsabteilung",
            "selection": {
                "filter": "item",
                "values": [
                    "10-33", "41-43", "49-53", "55-56",
                    "58-63", "62-63", "64-66", "68-75",
                    "77+79-82", "84", "85", "86-88", "90-96"
                ]  # All industry divisions except aggregates (5-96, 5-43, 45-96, 45-47)
            }
        }
    ],
    "response": {"format": "json-stat2"}
}

response = requests.post(API_URL, json=query)
response.raise_for_status()

dataset_json = response.json()
print("API request successful.")
print("Dimensions returned:", dataset_json["id"])

API request successful.
Dimensions returned: ['Offene Stellen', 'Wirtschaftsabteilung', 'Quartal']


### Step 3: Convert JSON-stat to DataFrame <a id="step3"></a>

Dimension labels and vacancy values are extracted from the JSON-stat2 response. A cartesian product of all dimension combinations is built to reconstruct the full dataset as a flat DataFrame.

In [19]:
# Convert JSON-stat2 to DataFrame

value_list = dataset_json["value"]
dim_ids = dataset_json["id"]
dim_sizes = dataset_json["size"]
dimension = dataset_json["dimension"]

# Create category label lists for all dimensions
category_labels = []
for dim in dim_ids:
    labels = dimension[dim]["category"]["label"]
    category_labels.append(list(labels.values()))

# Build cartesian product of all dimensions
combinations = list(itertools.product(*category_labels))

# Create dataframe
df_industry = pd.DataFrame(combinations, columns=dim_ids)
df_industry["vacancies"] = value_list

# Rename columns to clean names
df_industry = df_industry.rename(columns={
    "Offene Stellen": "metric",
    "Wirtschaftsabteilung": "industry",
    "Quartal": "quarter"
})

# Drop the metric column — we only requested total vacancies so it's redundant
df_industry = df_industry.drop(columns=["metric"])

print("Raw dataset shape:", df_industry.shape)
print("\nIndustries:", df_industry["industry"].unique())
df_industry.head()

Raw dataset shape: (1755, 3)

Industries: ['10-33 Manufactury' '41-43 Construction' '49-53 Transport and stockage'
 '55-56 Hotels and gastronomy' '58-63 IT and communications'
 '62-63 IT and other information services' '64-66 Finance and assurance'
 '68-75 Real estate and scientific service activities'
 '77+79-82 Administrative and support service activities (without 78)'
 '84 Public administration' '85 Education'
 '86-88 Health and social work activities' '90-96 Arts and amusement']


,industry,quarter,vacancies
0,10-33 Manufactury,1992Q2,NaN
1,10-33 Manufactury,1992Q3,NaN
2,10-33 Manufactury,1992Q4,NaN
3,10-33 Manufactury,1993Q1,NaN
4,10-33 Manufactury,1993Q2,NaN


### Step 4: Save Raw Dataset <a id="step4"></a>

The unmodified parsed data is saved as a raw CSV before any cleaning is applied.

In [8]:
raw_output_path = DATA_RAW / "bfs_vacancies_economic_division_raw.csv"
df_industry.to_csv(raw_output_path, index=False)

print("Raw dataset saved to:", raw_output_path)

Raw dataset saved to: ../data/raw/bfs_vacancies_economic_division_raw.csv


### Step 5: Data Cleaning <a id="step5"></a>

Columns are renamed for readability, the quarter field is converted to pandas Period format, and industry labels are standardised using a controlled mapping to align with BFS NOGA category names.

In [20]:
df_industry_clean = df_industry.copy()

# Standardize industry labels 
industry_map = {
    "10-33 Manufactury":                                                    "10-33 Manufacturing",
    "41-43 Construction":                                                   "41-43 Construction",
    "49-53 Transport and stockage":                                         "49-53 Transport and storage",
    "55-56 Hotels and gastronomy":                                          "55-56 Hospitality",
    "58-63 IT and communications":                                          "58-63 ICT",
    "62-63 IT and other information services":                              "62-63 IT services",
    "64-66 Finance and assurance":                                          "64-66 Finance and Insurance",
    "68-75 Real estate and scientific service activities":                  "68-75 Real estate & professional services",
    "77+79-82 Administrative and support service activities (without 78)":  "77+79-82 Administrative services",
    "84 Public administration":                                             "84 Public administration",
    "85 Education":                                                         "85 Education",
    "86-88 Health and social work activities":                              "86-88 Health and social work",
    "90-96 Arts and amusement":                                             "90-96 Arts and amusement"
}

df_industry_clean["industry"] = df_industry_clean["industry"].map(industry_map)

print("Industry labels after mapping:")
print(df_industry_clean["industry"].unique())

Industry labels after mapping:
['10-33 Manufacturing' '41-43 Construction' '49-53 Transport and storage'
 '55-56 Hospitality' '58-63 ICT' '62-63 IT services'
 '64-66 Finance and Insurance' '68-75 Real estate & professional services'
 '77+79-82 Administrative services' '84 Public administration'
 '85 Education' '86-88 Health and social work' '90-96 Arts and amusement']


### Step 6: Filter to Recent Quarters <a id="step6"></a>

The dataset is filtered to the most recent 8 quarters to align with the micro-level job postings timeframe.

In [24]:
# Convert quarter to Period and filter to last 8 quarters
df_industry_clean["quarter"] = pd.PeriodIndex(df_industry_clean["quarter"], freq="Q")

latest_quarters = sorted(df_industry_clean["quarter"].unique())[-8:]
df_industry_clean = df_industry_clean[df_industry_clean["quarter"].isin(latest_quarters)].copy()

print("Quarters kept:", sorted(df_industry_clean["quarter"].unique()))
print("Shape after filter:", df_industry_clean.shape)

Quarters kept: [Period('2024Q1', 'Q-DEC'), Period('2024Q2', 'Q-DEC'), Period('2024Q3', 'Q-DEC'), Period('2024Q4', 'Q-DEC'), Period('2025Q1', 'Q-DEC'), Period('2025Q2', 'Q-DEC'), Period('2025Q3', 'Q-DEC'), Period('2025Q4', 'Q-DEC')]
Shape after filter: (104, 3)


In [25]:
# Sort Final Dataset

df_industry_clean = df_industry_clean.sort_values(
    by=["quarter", "industry"]
).reset_index(drop=True)

### Step 7: Validation <a id="step7"></a>

Final shape, quarter range, unique industry labels, and missing values are printed to confirm the dataset is complete and correctly formatted before export.

In [26]:
print("Final dataset shape:", df_industry_clean.shape)
print("\nQuarter range:")
print(df_industry_clean["quarter"].min(), "to", df_industry_clean["quarter"].max())

print("\nUnique industries:")
print(sorted(df_industry_clean["industry"].unique()))

print("\nMissing values:")
print(df_industry_clean.isna().sum())

Final dataset shape: (104, 3)

Quarter range:
2024Q1 to 2025Q4

Unique industries:
['10-33 Manufacturing', '41-43 Construction', '49-53 Transport and storage', '55-56 Hospitality', '58-63 ICT', '62-63 IT services', '64-66 Finance and Insurance', '68-75 Real estate & professional services', '77+79-82 Administrative services', '84 Public administration', '85 Education', '86-88 Health and social work', '90-96 Arts and amusement']

Missing values:
industry     0
quarter      0
vacancies    0
dtype: int64


### Step 8: Export Clean Dataset <a id="step8"></a>

The cleaned dataset is saved to `data/external/bfs_vacancies_economic_division_clean.csv` for use in the Phase 3 industry merge.

In [27]:
final_output_path = DATA_PROCESSED / "bfs_vacancies_economic_division_clean.csv"
df_industry_clean.to_csv(final_output_path, index=False)

print("Clean dataset saved to:", final_output_path)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Clean dataset saved to: ../data/processed/bfs_vacancies_economic_division_clean.csv
